# Tethys-Chloris (T&C): spatially-distributed modeling

## Material
* Case Study Materials
  * Script to create the model inputs (`create_data_2d.jl`)

* Input Data
  * Files containing CO2 air concentration
  * Digital elevation model and catchment geodata (`GeoExample.mat`)
  * Meteorological station data from the first exercise (`Data_CH-Cha_run.mat`)

* Visualization Tools
  * Functions for results visualizations (Plots.jl)

## Objectives

This session is designed to help students learn how to prepare input files and execute a basic distributed simulation using the spatially-distributed (2D) version of the T&C model. It extends the plot-scale workflows of the previous exercises to a small catchment, where terrain, flow routing and spatially varying inputs become relevant.

The tutorial here will focus on one virtual catchment located at [Chamau, ZG](https://www.google.com/maps/place/47%C2%B012'36.8%22N+8%C2%B024'38.3%22E/@47.2102278,8.4099778,273m/data=!3m2!1e3!4b1!4m4!3m3!8m2!3d47.2102269!4d8.4106451?entry=ttu), using the same meteorological data as in the first exercise.

## Environment setup

Run this cell first whenever you start or restart the kernel. It activates the project in the parent folder and installs missing dependencies; the first run may take a few minutes. Keep the repository folder structure unchanged when sharing this notebook.


In [ ]:
import Pkg

# Activate the shared project environment using a relative path.
Pkg.activate(normpath(joinpath(@__DIR__, "..")))

# Install any missing dependencies listed in Project.toml.
Pkg.instantiate()

println("Active Julia environment: ", Base.active_project())

# Part A: input preparation, model setup and simulation

## 1. Prepare the observation-based spatial inputs

In addition to the meteorological input data prepared in the previous exercise, the spatial (2D) application needs topographic information for every computation cell: elevation, slope, aspect, flow directions, the stream network, land cover and soil texture.

All this information is bundled in the file **`data/GeoExample.mat`**. Load it and inspect its contents:

In [ ]:
using MAT: matread

geo = matread(joinpath(@__DIR__, "data", "GeoExample.mat"))
println(sort(collect(keys(geo))))

Reflect on the study site and the catchment characteristics:

- *What is the size of the topography? How many computation cells does it have?*
- *What is the spatial resolution?*
- *How many types of vegetation are there? Check the variable `ksv`. Remember that the demo combines codes 1–4 into two classes before the simulation. Search for `ksv` in `create_data_2d.jl` to see how the original codes are grouped.*
- *What are the soil properties in the domain? (check `PSAN`, `PCLA`, `PORG`)*

## 2. Understand the case study

Each grid cell is assigned a land cover class through `ksv`. In `create_data_2d.jl`, original code 1 remains class 1 (wood), while codes 2-4 are grouped into class 2 (grass/shrub):

- **Class 1:** 100% vegetation cover, with high vegetation (wood) and a high-vegetation root depth (`ZR95`) of 400 mm.
- **Class 2:** 100% vegetation cover, with low vegetation (grass/shrub) and a low-vegetation root depth (`ZR95`) of 250 mm.

The script uses `vegetation_indices` to select the first or second entry of the parameter arrays for each class and writes two parameter blocks, `"1"` and `"2"`, to `data/exercise_2d_parameters.yaml`. This file is generated when you run the input-preparation script in Section 4.

During model initialization, each cell's reclassified `ksv` selects the matching YAML parameter block and the corresponding initial-state group in the NetCDF file. Thus, `ksv` determines where each set of vegetation properties is applied across the catchment.

Before running the simulation, answer the following questions:

- *What simulation period and time step are specified in `create_data_2d.jl`? Find `simulation_start` and `simulation_end` near the top of the script, then locate `timerange` and `dt` in the `simulation` parameter block. The time step `dt` is given in seconds.*
- *What root depths and initial Leaf Area Index (LAI) values are assigned to wood and grass/shrub? Search for `ZR95` and `"LAI"` in `create_data_2d.jl` and identify the values used for each land cover class.*
- *Where do you expect the wettest and driest cells of the catchment to be, and why? Think about how topography influences the distribution of water and energy across the catchment.*

## 3. Understand the structure of the model

The 2D model uses the same local soil, vegetation and energy-balance processes as the plot-scale model, while adding interactions and spatial differences across the catchment. The following guide explains these additions and where to inspect them. Source-code paths below are relative to the repository root, one level above this notebook.

- **Flow routing (`T_flow`).** The routing matrix in `GeoExample.mat` describes connections between grid cells. The model uses these connections to redistribute surface and subsurface water towards the catchment outlet. Search for `T_flow` in `create_data_2d.jl` to see how the matrix is stored in the input NetCDF file.

- **Spatially distributed forcing (elevation gradients).** Station measurements are adjusted for differences between each cell's elevation (`DTM`) and the reference elevation (`Zbas`). Search for `Zbas` in `create_data_2d.jl`, then open `TethysChloris.jl/src/Model.jl` and search for `calculate_spatial_forcing` and `_apply_lapse_rate`. **The gradients are currently hard-coded in this model function**, rather than exposed as settings in the input-preparation script or YAML file. For example, air temperature uses -0.0045 degrees C per metre; wind speed, dew-point temperature and direct-radiation components have their own gradients. The function applies these adjustments when preparing the forcing for each time step.

- **Terrain effects on radiation (`SvF` and `ShF`).** The sky view factor `SvF` describes how much of the sky is visible from a cell and affects diffuse radiation. Search for `sky_view_factor` in `TethysChloris.jl/src/spatial/GridMetadata.jl`. The shadow factor `ShF` accounts for terrain blocking direct sunlight as the solar position changes. Search for `calculate_spatial_forcing` in `TethysChloris.jl/src/Model.jl` to see where it is calculated and used.

- **Spatial output timing (`tstore`).** This setting is in the simulation cells in **Section 4 of this notebook**, inside `SpatialRunOptions(...)` passed to `run_simulation`. The short run uses `tstore = [10]`; the optional full run uses `tstore = [40, 2160]`. These are time-step indices at which the model saves maps summarizing the interval since the previous spatial save: averages or accumulated sums, depending on the variable. They are not instantaneous snapshots. The final step is included automatically. To inspect this behavior, search for `tstore` and `save_outputs!` in `TethysChloris.jl/src/spatial/output_manager_2d.jl`.

- **Saving results.** Results are written during the simulation to a single NetCDF file specified by `output_filename` in `SpatialRunOptions(...)`. The main groups are `spatial_mean` and `spatial_std` (catchment time series), `point_1` (tracked-cell time series), `time_average` (spatial maps summarized over output intervals), and `grid` (static spatial information). Section 5 shows how to read and plot these outputs.


## 4. Ready to run the simulation

The script **`create_data_2d.jl`** prepares two model input files from the supplied data: a NetCDF file containing meteorological forcing, spatial grid information and initial conditions, and a YAML file containing parameters for both land cover classes. After generating these files, open `data/exercise_2d_parameters.yaml` and set the bedrock conductivity `Kbot` to `0.11` mm/h in the `rock:` block of **both** land cover classes. Save the YAML file before initializing the model.

Run the following cell to generate the input files:


In [ ]:
include(joinpath(@__DIR__, "create_data_2d.jl"))

Follow the same steps as in exercise 1 (section 6) to initialize the model, this time with the 2D model type:

In [ ]:
using NCDatasets
using YAML
using Roots
using TethysChloris

FT = Float64
data_dir = joinpath(@__DIR__, "data")

netcdfpath = joinpath(data_dir, "exercise_2d_data.nc")
yamlpath = joinpath(data_dir, "exercise_2d_parameters.yaml")

model, forcing = initialize_model(Model2D{FT}, netcdfpath, yamlpath);

In [ ]:
options = ModelOptions(
    SoilBiogeochemistry = false,
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
);

During the exercise session, run the simulation for a short period first to validate the setup. Ten timesteps take well under a minute:

In [ ]:
diagnostics, manager = run_simulation(
    model,
    forcing;
    options = options,
    run_options = SpatialRunOptions(
        verbose = true, # Print progress at every 2D time step.
        NN = 10,
        output_level = TethysChloris.AllOutputs(),
        output_filename = joinpath(data_dir, "smoke_results.nc"),
        compress = true,
        parallel = false,
        points = [(1, 1)], # Outlet grid cell (row, column).
        tstore = [10],
    ),
);

The full simulation covers 1 May to 3 October 2005 with an hourly time step and may take several hours. The optional full-run code below is commented out; students only need to run the short test above. Precomputed full-run results are provided for the visualizations in Section 5.


In [ ]:
# OPTIONAL FULL RUN: uncomment the code below only if you want to rerun the full simulation.
# println("Julia compute threads: ", Threads.nthreads())
# if Threads.nthreads() == 1
#     @warn "Only one compute thread is available. Set Julia: Num Threads and restart the kernel for parallel execution."
# end

# model, forcing = initialize_model(Model2D{FT}, netcdfpath, yamlpath);

# diagnostics, manager = run_simulation(
#     model,
#     forcing;
#     options = options,
#     run_options = SpatialRunOptions(
#         verbose = true, # Print progress at every 2D time step.
#         output_level = TethysChloris.AllOutputs(),
#         output_filename = joinpath(data_dir, "exercise_2d_results.nc"),
#         compress = true,
#         parallel = true,
#         points = [(1, 1)], # Outlet grid cell (row, column).
#         tstore = [40, 2160],
#     ),
# );


## 5. Checking the results

Read the output NetCDF file to inspect catchment time series, spatial maps and conditions at the tracked grid cell. The main output groups are:

| Group | Content |
|---|---|
| `spatial_mean` / `spatial_std` | Catchment-average time series and spatial standard deviations |
| `point_1` | Time series at the tracked grid cell |
| `time_average` | Spatial maps of interval averages or accumulated sums, depending on the variable, saved at the `tstore` steps |
| `grid` | Static spatial information, including elevation and land cover |

Use the provided full-simulation results in **`data/exercise_2d_results.nc`** to interpret seasonal changes. The cell below loads this file for visualization, so you do not need to rerun the full simulation. The 10-step test above saves a separate file, `smoke_results.nc`.


In [ ]:
# Load the provided full-simulation results for visualization.
results_filename = "exercise_2d_results.nc"
results = NCDataset(joinpath(data_dir, results_filename))
sm = results.group["spatial_mean"]
ta = results.group["time_average"]
pt = results.group["point_1"]
nothing

### 5.1 Time series of spatially-averaged variables

In [ ]:
using Plots

days = collect(0:length(Array(sm["Ts"])) - 1) ./ 24

p1 = plot(days, Array(sm["Ts"]), label = "Ts", ylabel = "Surface temperature [°C]", legend = :best)
p2 = plot(days, Array(sm["QE"]), label = "QE", ylabel = "Latent heat [W/m²]", legend = :best)
p3 = plot(days, Array(sm["H"]), label = "H", ylabel = "Sensible heat [W/m²]", legend = :best)
p4 = plot(days, Array(sm["ZWT"]), label = "ZWT", ylabel = "Water table depth [mm]", legend = :best)
plot(p1, p2, p3, p4, layout = (4, 1), size = (800, 900), xlabel = "Time [days]")

### 5.2 Spatial maps

Each interval map shows the time average between successive `tstore` save points, not an instantaneous value at one time step; the first interval covers the start of the run up to the first save point. For `V`, `layer = 1` selects the uppermost soil layer (0-10 mm in this setup).


In [ ]:
Ts_maps = Array(ta["Ts"]) # (m_cell, n_cell, snapshots)
snapshots = ["output interval $i" for i in 1:size(Ts_maps, 3)]

plots = [heatmap(Ts_maps[:, :, i]; title = "Ts, $(snapshots[i])", clims = (0, 30), aspect_ratio = 1, xflip = false, yflip = false,
                 xlabel = "Grid column", ylabel = "Grid row")
         for i in 1:size(Ts_maps, 3)]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

In [ ]:
# Air temperature: higher terrain is cooler because the lapse rate is -0.0045 degrees C/m.
Ta_maps = Array(ta["Ta"]) # (m_cell, n_cell, output intervals)

# Use the data range, with the same temperature scale for all intervals.
ta_values = filter(isfinite, collect(skipmissing(vec(Ta_maps))))
@assert !isempty(ta_values) "No finite air-temperature values are available."
ta_limits = extrema(ta_values)

Ta_plots = [
    heatmap(Ta_maps[:, :, i];
        title = "Air temperature, interval $i",
        colorbar_title = "Temperature [°C]",
        clims = ta_limits,
        aspect_ratio = 1,
        xflip = false,
        yflip = false,
        xlabel = "Grid column",
        ylabel = "Grid row",
    ) for i in 1:size(Ta_maps, 3)
]
plot(Ta_plots...; layout = (1, length(Ta_plots)), size = (400 * length(Ta_plots), 350))


In [ ]:
V_maps = Array(ta["V"]) # (m_cell, n_cell, layers, snapshots)
layer = 1

plots = [heatmap(V_maps[:, :, layer, i]; title = "V (layer $layer), $(snapshots[i])", aspect_ratio = 1, xflip = false, yflip = false,
                 xlabel = "Grid column", ylabel = "Grid row", colorbar_title = "Liquid water storage [mm]")
         for i in 1:size(V_maps, 4)]
plot(plots...; layout = (1, length(plots)), size = (300 * length(plots), 300))

In [ ]:
# This setup has one crown area per cell with 100% vegetation cover.
# Add high- and low-vegetation LAI; the inactive vegetation layer contributes zero.
LAI_H_maps = Array(ta["LAI_H"]) # (m_cell, n_cell, crownareas, output intervals)
LAI_L_maps = Array(ta["LAI_L"])
@assert size(LAI_H_maps, 3) == size(LAI_L_maps, 3) == 1 "This plot assumes one crown area per cell."
LAI_maps = LAI_H_maps[:, :, 1, :] .+ LAI_L_maps[:, :, 1, :]

# Use the same LAI scale for all output intervals.
lai_values = filter(isfinite, collect(skipmissing(vec(LAI_maps))))
@assert !isempty(lai_values) "No finite LAI values are available."
lai_limits = (0.0, max(1.0, maximum(lai_values)))

LAI_plots = [
    heatmap(LAI_maps[:, :, i];
        title = "LAI, interval $i",
        colorbar_title = "LAI [m²/m²]",
        clims = lai_limits,
        aspect_ratio = 1,
        xflip = false,
        yflip = false,
        xlabel = "Grid column",
        ylabel = "Grid row",
    ) for i in 1:size(LAI_maps, 3)
]
plot(LAI_plots...; layout = (1, length(LAI_plots)), size = (400 * length(LAI_plots), 350))


### 5.3 Liquid water storage at the tracked point

The tracked point is currently the outlet grid cell (row 1, column 1); change `points` in `SpatialRunOptions(...)` to select a different cell.


In [ ]:
V = Array(pt["V"]) # (time, layers)

# Read the actual layer boundaries [mm] for the tracked cell's land cover class.
profile_parameters = YAML.load_file(yamlpath)[string(pt.attrib["ksv"])]
depth_edges = Float64.(profile_parameters["simulation"]["Zs"])
@assert length(depth_edges) == size(V, 2) + 1 "Soil boundaries must match the output layers."
@assert all(diff(depth_edges) .> 0) "Soil depths must increase downwards."
@assert length(days) == size(V, 1) "Time coordinates must match the output records."

# Supply bin edges on both axes so each layer has its actual thickness.
dt_days = profile_parameters["simulation"]["dt"] / 86400
time_edges = vcat(days .- dt_days / 2, days[end] + dt_days / 2)

heatmap(time_edges, depth_edges, V';
    xlabel = "Time [days]",
    ylabel = "Soil depth [mm]",
    yflip = true,
    ylims = (first(depth_edges), last(depth_edges)),
    yticks = 0:100:last(depth_edges),
    colorbar_title = "Liquid water storage [mm]",
    title = "Liquid water storage at point 1",
)


- *During which periods does the profile dry from the top down, and when is it rewetted?*
- *Try plotting a different layer or a different variable (e.g. `Tdp`, the soil temperature profile) at the tracked point.*

In [ ]:
close(results)

### Find out more

[Laboratory of Catchment Hydrology and Geomorphology](https://www.epfl.ch/labs/change/)
[T&C source code (Julia)](github.com/CHANGE-EPFL/TethysChloris.jl)
[T&C source code (MATLAB)](https://github.com/simonefatichi/TeC_Source_Code)


### References 

Fatichi, S., Ivanov, V. Y., & Caporali, E. (2012). A mechanistic ecohydrological model to investigate complex interactions in cold and warm water‐controlled environments: 1. Theoretical framework and plot‐scale analysis. *Journal of Advances in Modeling Earth Systems, 4*(2).

Remondi, F., Botter, M., Burlando, P., & Fatichi, S. (2019). Variability of transit time distributions with simple catchments: the role of the spatial distribution of rainfall. *Hydrology and Earth System Sciences, 23*(8).